# 03 — Pemeriksaan Hasil Embedding

**Tahap 6 ceklis: Embedding** · Protokol DSIC-2906-1 bagian 19 butir 3; README bagian 12, 28 (E1)

**Model beku:** `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`, revisi `e8f8c211226b894fcb81acc59f3b34ba3efd5f42`, dimensi 384, dinormalisasi, metrik cosine. Dihasilkan oleh `python -m src.embedding.embed` (GPU mps, 153 detik); checksum di `data/manifests/vector_manifest.csv`.

**Tujuan:** memastikan vektor tidak rusak, urutannya sesuai korpus, dan tetangga terdekatnya masuk akal sebelum dipakai untuk ground truth dan indeks DiskANN.

## Langkah 1: Membaca vektor, doc_id, dan korpus

Baris ke-*i* matriks vektor berpasangan dengan baris ke-*i* `base_ids.parquet`. DiskANN nanti hanya mengenal nomor baris, sehingga urutan ini wajib konsisten.

In [1]:
# Vektor hasil embedding, urutan doc_id-nya, dan korpus untuk menampilkan teks
import numpy as np
import pandas as pd

vectors = np.load("../data/embeddings/base_vectors.npy")
ids = pd.read_parquet("../data/embeddings/base_ids.parquet")
corpus = pd.read_parquet("../data/processed/corpus.parquet")

print("Bentuk matriks:", vectors.shape, "| tipe:", vectors.dtype)

Bentuk matriks: (64354, 384) | tipe: float32


## Langkah 2: Pemeriksaan kualitas vektor

Empat pemeriksaan: tidak ada nilai NaN/inf, semua norma = 1 (syarat agar cosine = perkalian titik), urutan `doc_id` sama dengan korpus, dan jumlah vektor identik.

In [2]:
# Jumlah nilai rusak (NaN/inf), panjang (norma) vektor, dan kesesuaian urutan doc_id
print("Nilai NaN/inf:", (~np.isfinite(vectors)).sum())

norma = np.linalg.norm(vectors, axis=1)
print("Norma min / maks:", norma.min().round(5), "/", norma.max().round(5))

print("Urutan doc_id sama dengan korpus:", (ids["doc_id"].values == corpus["doc_id"].values).all())

# Vektor yang identik dengan vektor lain (berasal dari teks yang persis sama)
_, idx_unik = np.unique(vectors, axis=0, return_index=True)
print("Vektor duplikat:", len(vectors) - len(idx_unik))

Nilai NaN/inf: 0
Norma min / maks: 1.0 / 1.0
Urutan doc_id sama dengan korpus: True
Vektor duplikat: 772


## Langkah 3: Tetangga terdekat (uji kewajaran)

Lima dokumen dengan cosine tertinggi terhadap satu dokumen acuan, dihitung secara eksak terhadap seluruh 64.354 vektor. Tiga dokumen acuan diuji: atraksi (ke-0), berita (ke-25000), dan desa wisata (ke-18000).

In [3]:
# Lima dokumen paling mirip dengan dokumen ke-i; karena vektor ternormalisasi, cosine = perkalian titik
def tetangga_terdekat(i, k=5):
    skor = vectors @ vectors[i]
    urutan = np.argsort(-skor)[1:k + 1]
    hasil = corpus.iloc[urutan][["source_type", "text"]].copy()
    hasil["cosine"] = skor[urutan].round(3)
    return hasil

print("Dokumen acuan:", corpus.iloc[0]["text"])
tetangga_terdekat(0)

Dokumen acuan: Wisata Bunga Rafflesia Wisata alam. Data Sungai Taleh, Nagari Paninjauan, Kecamatan Tanjung Raya Fasilitas: Kamar Mandi Umum; Musholla; Tempat makan.


,source_type,text,cosine
13340,jadesta_atraksi,"Tari cepetan Wisata budaya. Desa Gunungsari, K...",0.892
13480,jadesta_atraksi,"Tari ebleg Wisata budaya. Desa Gunungsari, Kar...",0.889
135,jadesta_atraksi,Nangkap Kepiting Bakau Dan Ranjungan Wisata al...,0.887
13479,jadesta_atraksi,"Tari Lengger Wisata budaya. Desa Gunungsari, K...",0.886
14733,jadesta_atraksi,Edu Wisata Batik Kaganga Wisata budaya. Jalan ...,0.883


In [4]:
# Tetangga terdekat untuk dokumen ke-25000
print("Dokumen acuan:", corpus.iloc[25000]["text"])
tetangga_terdekat(25000)

,source_type,text,cosine
24810,news,"Baru Diresmikan, Jembatan Gantung Pongpet Anjlok",0.722
26271,news,Kodim 0108/Agara dan Warga Gotong Royong Pasan...,0.689
26565,news,Jembatan Gantung Rawayan Tarumanagara yang eks...,0.645
41254,news,Jembatan Batu Bacan di Halmahera Selatan,0.643
33904,news,"Jalur Trekking Cisadon Dibangun Loket Baru, Pe...",0.628


In [5]:
# Tetangga terdekat untuk dokumen ke-18000
print("Dokumen acuan:", corpus.iloc[18000]["text"])
tetangga_terdekat(18000)

,source_type,text,cosine
15826,jadesta_desa,"Desa Wisata Pangirkiran desa pangirkiran,kecam...",0.926
18001,jadesta_desa,"Desa Wisata Padang Bulan desa padang bulan, ke...",0.925
17174,jadesta_desa,Desa Wisata Tambakmekar Desa Tambamekar Fasili...,0.925
17952,jadesta_desa,"Desa Wisata Pagarantonga Desa Pagarantonga, Ke...",0.925
18784,jadesta_desa,"Desa Wisata Babakan Babakan, Poncosari, Standa...",0.922


## Langkah 4: Contoh pencarian berfilter eksak (bentuk sederhana ground truth G0)

Kueri Q1 dari README bagian 15: *"wisata bahari untuk melihat lumba-lumba"* dengan filter `kabupaten = Kabupaten Tanggamus`. Kueri hanya dibandingkan dengan titik yang lolos filter (eligible), lalu diambil 10 terdekat. Hasil semacam inilah yang menjadi "kunci jawaban" untuk menilai Recall@10 PostFilter, TwoHop, dan AdaptiveL.

In [6]:
# Model yang sama untuk mengubah kueri menjadi vektor
from sentence_transformers import SentenceTransformer
import yaml

cfg = yaml.safe_load(open("../configs/embedding.yaml"))
model = SentenceTransformer(cfg["model_name"], revision=cfg["model_version"])

# Contoh kueri dari README bagian 15 (Q1): kueri semantik + filter kabupaten
kueri = "wisata bahari untuk melihat lumba-lumba"
filter_kab = "Kabupaten Tanggamus"

q = model.encode([kueri], normalize_embeddings=True)[0]
skor = vectors @ q

# Pencarian eksak berfilter: hanya titik yang memenuhi predikat yang dibandingkan
eligible = corpus["kabupaten"].apply(lambda daftar: filter_kab in daftar).values
print("Titik eligible:", eligible.sum(), f"({eligible.mean() * 100:.3f}%)")

idx = np.flatnonzero(eligible)
top10 = idx[np.argsort(-skor[idx])[:10]]
hasil = corpus.iloc[top10][["source_type", "text"]].copy()
hasil["cosine"] = skor[top10].round(3)
hasil

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Titik eligible: 110 (0.171%)


,source_type,text,cosine
187,jadesta_atraksi,Dolphin Tour Wisata alam. Perairan teluk Kilua...,0.641
24296,news,Objek Wisata Pantai Pasir Putih Geseran Naga R...,0.562
24257,news,Ini Destinasi Wisata Pantai Pasir Putih PIK 2 ...,0.539
33681,news,"Selain Pariwisata, Pesisir Selatan menyimpan w...",0.480
4064,jadesta_atraksi,Explore pulau kelapa Wisata alam. Pulau kelapa...,0.449
404,jadesta_atraksi,Wisata Pantai Pasir Putih Wisata alam. Pantai ...,0.427
6212,jadesta_atraksi,Terbang Bebas di Langit Sukaraja : Avitourism ...,0.417
12907,jadesta_atraksi,Pencak silat PSHT Wisata budaya. Desa Wisata S...,0.400
4301,jadesta_atraksi,Berwisata di Air Terjun Way Tebing Wisata alam...,0.397
4421,jadesta_atraksi,Jungle Tracking Air Terjun Sukaraja Wisata ala...,0.387


## Ringkasan hasil

| Pemeriksaan | Hasil |
|---|---|
| Bentuk matriks | 64.354 × 384, float32 |
| Nilai NaN/inf | 0 |
| Norma min / maks | 1,0 / 1,0 |
| Urutan doc_id = korpus | Ya |
| Vektor identik (duplikat) | 772 |
| Titik eligible filter Tanggamus | 110 (0,171%, bin B2) |
| Hasil teratas kueri lumba-lumba | *Dolphin Tour*, perairan Teluk Kiluan, Tanggamus (cosine 0,641) |

**Tetangga terdekat:**
- **Berita (ke-25000, jembatan gantung di Jember):** tetangganya berita lain tentang jembatan gantung (cosine 0,63–0,72). Masuk akal secara makna.
- **Atraksi (ke-0, Wisata Bunga Rafflesia, Agam):** tetangganya tarian di Desa Gunungsari dan wisata batik (cosine 0,88–0,89), **tidak mirip secara makna**.
- **Desa wisata (ke-18000, Desa Wisata Lantosan):** tetangganya desa wisata lain dengan pola teks yang sama (cosine 0,92–0,93).

## Kesimpulan dan implikasi (draf)

1. **Vektor valid secara teknis** dan siap dipakai untuk ground truth dan indeks DiskANN.
2. **Kemiripan dokumen JADESTA didominasi pola teks templat**, bukan makna. Teks atraksi/desa disusun dari nama + "Wisata {kategori}." + alamat + daftar fasilitas (protokol bagian 16 langkah 6). Daftar fasilitas yang seragam ("Kamar Mandi Umum; Musholla; Tempat makan") membuat dokumen yang tidak berhubungan terlihat sangat mirip (cosine ~0,9). Ini perlu dibahas saat bimbingan: README bagian 9 menyarankan hanya teks yang mengandung konten semantik yang di-embed, dan tidak memasukkan metadata secara berlebihan ke teks. Objek ilmiah penelitian adalah perilaku pencarian (bukan kualitas relevansi), tetapi struktur ketetanggaan graf tetap ikut terpengaruh.
3. **Kesalahan wilayah berita tidak hanya terjadi pada berita multi-kabupaten.** Pada hasil filter Tanggamus muncul berita *"Pantai Pasir Putih PIK 2 ... di Tangerang"* yang hanya tertaut ke satu kabupaten (Tanggamus), karena nama atraksinya sama. Tanda `region_ambiguous` tidak menangkap kasus seperti ini, sehingga perlakuan "keluarkan berita multi-kabupaten" saja belum cukup.
4. **772 vektor identik** (teks yang persis sama) dapat menghasilkan skor seri pada ground truth; aturan pemecah seri (misalnya urutan nomor baris) perlu ditetapkan saat membangun G0.